In [12]:
import os
import sys

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "..", "..", ".."))

sys.path.append(project_root)

In [2]:
from src.citibike.citibike_utils import get_trip_duration_mins
from src.utils.datetime_utils import timestamp_to_date_col
from pyspark.sql.functions import create_map, lit

In [3]:
df = spark.read.table('citibike_dev.01_bronze.jc_citibike')

In [5]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [6]:
df = timestamp_to_date_col(spark, df, 'started_at', 'trip_start_date')

In [7]:
df = df.withColumn("metadata",
            create_map(
                lit("pipeline_id"), lit("placeholder"),
                lit("run_id"), lit("placeholder"),
                lit("task_id"), lit("placeholder"),
                lit("processed_timestamp"), lit("placeholder")
            ))

In [9]:
df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "metadata"
)

In [10]:
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable("citibike_dev.02_silver.jc_citibike")